# Compute Imagery Statistics for Sentinel-2 Composites

This notebook computes statistics for each site, monthly composite, and index band in the Step 0 imagery outputs. Each index is summarized using finite pixels inside the rasterized site polygon footprint.

### Indices Analyzed
- **NDVI** (Normalized Difference Vegetation Index): $(B08 - B04) / (B08 + B04)$
- **GNDVI** (Green Normalized Difference Vegetation Index): $(B08 - B03) / (B08 + B03)$
- **SAVI** (Soil-Adjusted Vegetation Index, $L=0.428$): $((B08 - B04) / (B08 + B04 + 0.428)) \times 1.428$
- **NDRE** (Normalized Difference Red Edge): $(B8A - B05) / (B8A + B05)$

### Statistics Computed
Each output row identifies a `site`, `year`, `month`, `window`, and `index`, and includes:
- **Pixel coverage**: `n_valid_pixels`, `site_area_pixels` (polygon-footprint denominator), and `pct_valid_pixels`.
- **Area**: `total_area_ha`, `total_area_km2`, `valid_area_ha`, and `valid_area_km2`.
- **Index distribution**: `mean`, sample standard deviation (`std`, `ddof=1`), and quartiles `p25`, `p50`, and `p75`; `p50` is the median.
- **Composite QA**: `n_scenes` and `mean_cloud` from [outputs/composites/qa_manifest.csv](outputs/composites/qa_manifest.csv).

The CSV is written to [outputs/composites/imagery_stats.csv](outputs/composites/imagery_stats.csv). Running the export cell replaces any existing file at that path.

In [1]:
import re
import sys
import warnings
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import rasterio

# Suppress internal numpy/rasterio shape deprecation warnings if present
warnings.filterwarnings("ignore", category=DeprecationWarning)

ROOT = Path.cwd()
if not (ROOT / "outputs").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

## 1 · Configuration and Paths

In [2]:
# Input and output directory locations
COMPOSITES_DIR = ROOT / "outputs" / "composites"
QA_MANIFEST_PATH = COMPOSITES_DIR / "qa_manifest.csv"
OUTPUT_CSV_PATH = COMPOSITES_DIR / "imagery_stats.csv"

# Spectral index bands to summarize (bands 9-12 in composite GeoTIFFs)
TARGET_INDICES: tuple[str, ...] = ("NDVI", "GNDVI", "SAVI", "NDRE")

# Filename pattern: <site>_<YYYY>-<MM>.tif
FILENAME_PATTERN = re.compile(r"^(?P<site>.+)_(?P<year>\d{4})-(?P<month>\d{2})\.tif$")

## 2 · Discovery and Site Footprint Reference

Each site polygon is reprojected to a composite's CRS and rasterized onto its grid using pixel-center inclusion (`all_touched=False`). The site area pixel count is therefore the number of pixel centers inside the polygon, independent of valid-data masks and consistent with the acquisition workflow.

In [3]:
def discover_composite_tifs(composites_dir: Path) -> list[Path]:
    """Find site composite GeoTIFFs matching the expected naming pattern."""
    return sorted(
        path for path in composites_dir.glob("*/*.tif")
        if FILENAME_PATTERN.match(path.name)
    )


def calculate_site_footprints(
    tif_paths: list[Path],
    site_vector_path: Path,
) -> tuple[dict[str, np.ndarray], dict[str, float]]:
    """Rasterize each site polygon on its composite grid and calculate pixel area."""
    import geopandas as gpd
    from rasterio.features import geometry_mask

    sites = gpd.read_file(site_vector_path)
    if sites.crs is None:
        raise ValueError(f"Site polygons have no CRS: {site_vector_path}")
    if "site_name" not in sites.columns:
        raise ValueError("Study-site layer must contain a 'site_name' field")

    site_geometries = {
        str(site): group.geometry.to_list()
        for site, group in sites.groupby("site_name", sort=False)
    }
    site_masks: dict[str, np.ndarray] = {}
    site_pixel_area: dict[str, float] = {}
    site_grids: dict[str, tuple[object, object, tuple[int, int]]] = {}

    for path in tif_paths:
        match = FILENAME_PATTERN.match(path.name)
        if match is None:
            continue
        site = match.group("site")
        if site not in site_masks:
            if site not in site_geometries:
                raise ValueError(f"No study-site polygon found for {site!r}")
            with rasterio.open(path) as src:
                projected = gpd.GeoSeries(site_geometries[site], crs=sites.crs).to_crs(src.crs)
                site_masks[site] = geometry_mask(
                    [geometry.__geo_interface__ for geometry in projected],
                    out_shape=src.shape,
                    transform=src.transform,
                    invert=True,
                )
                site_pixel_area[site] = float(
                    abs(src.transform.a * src.transform.e - src.transform.b * src.transform.d)
                )
                site_grids[site] = (src.crs, src.transform, src.shape)
        else:
            with rasterio.open(path) as src:
                crs, transform, shape = site_grids[site]
                if (src.crs, src.transform, src.shape) != (crs, transform, shape):
                    raise ValueError(f"Composite grid changed across dates for {site}: {path}")

    return site_masks, site_pixel_area


SITE_VECTOR_PATH = ROOT / "sites" / "StudySites.gpkg"
tif_files = discover_composite_tifs(COMPOSITES_DIR)
site_masks, site_pixel_area = calculate_site_footprints(tif_files, SITE_VECTOR_PATH)

print(f"Discovered {len(tif_files)} composite rasters across {len(site_masks)} sites:")
for site, mask in sorted(site_masks.items()):
    site_area_pixels = int(mask.sum())
    total_area_ha = site_area_pixels * site_pixel_area[site] / 10_000.0
    total_area_km2 = site_area_pixels * site_pixel_area[site] / 1_000_000.0
    print(
        f"  {site:<22}: {site_area_pixels:>6} pixels "
        f"({total_area_ha:>8.2f} ha | {total_area_km2:>6.3f} km2)"
    )

Discovered 196 composite rasters across 7 sites:
  AlAhrashProtectorate  :  33848 pixels (  338.09 ha |  3.381 km2)
  BesorStream           :  11318 pixels (  113.03 ha |  1.130 km2)
  HaBesorNationalPark   :  91755 pixels (  918.64 ha |  9.186 km2)
  HolotNizzanim         : 139638 pixels ( 1395.25 ha | 13.953 km2)
  HolotPalmahim         :  62192 pixels (  620.89 ha |  6.209 km2)
  NahalRubin            :   7860 pixels (   78.56 ha |  0.786 km2)
  WadiGaza              :  28413 pixels (  284.33 ha |  2.843 km2)


## 3 · Extract Statistics Across Sites, Dates, and Indices

In [4]:
def extract_imagery_statistics(
    tif_paths: list[Path],
    site_masks: dict[str, np.ndarray],
    site_pixel_area: dict[str, float],
    qa_manifest_path: Path,
    indices: tuple[str, ...] = TARGET_INDICES,
) -> pd.DataFrame:
    """Summarize each index within the rasterized site polygon footprint."""
    if not qa_manifest_path.exists():
        raise FileNotFoundError(f"Missing composite QA manifest: {qa_manifest_path}")

    qa_df = pd.read_csv(qa_manifest_path, dtype={"site": str, "window": str})
    required_qa_columns = {"site", "window", "status", "n_scenes", "mean_cloud"}
    missing_qa_columns = required_qa_columns.difference(qa_df.columns)
    if missing_qa_columns:
        raise ValueError(f"QA manifest is missing columns: {sorted(missing_qa_columns)}")
    qa_df["window"] = qa_df["window"].str.strip()
    if qa_df.duplicated(["site", "window"]).any():
        raise ValueError("QA manifest contains duplicate site/window rows")
    qa_lookup = qa_df.set_index(["site", "window"])

    records: list[dict[str, Any]] = []

    for path in tif_paths:
        match = FILENAME_PATTERN.match(path.name)
        if match is None:
            continue

        site = match.group("site")
        year = int(match.group("year"))
        month = match.group("month")
        window = f"{year}-{month}"
        if site not in site_masks:
            raise ValueError(f"No rasterized polygon footprint for {site!r}")
        if (site, window) not in qa_lookup.index:
            raise ValueError(f"No QA manifest row for {site}, {window}")

        qa_info = qa_lookup.loc[(site, window)]
        if qa_info["status"] != "ok":
            raise ValueError(f"QA status is not ok for {site}, {window}: {qa_info['status']}")

        footprint = site_masks[site]
        pixel_area_m2 = site_pixel_area[site]
        site_area_pixels = int(footprint.sum())
        total_area_ha = site_area_pixels * pixel_area_m2 / 10_000.0
        total_area_km2 = site_area_pixels * pixel_area_m2 / 1_000_000.0

        with rasterio.open(path) as src:
            if src.shape != footprint.shape:
                raise ValueError(f"Raster and site mask shapes differ: {path}")
            descriptions = list(src.descriptions)
            missing_indices = [name for name in indices if name not in descriptions]
            if missing_indices:
                raise ValueError(f"Missing index bands {missing_indices} in {path}")
            band_numbers = [descriptions.index(name) + 1 for name in indices]
            data = src.read(band_numbers, masked=True)

        for band_position, index_name in enumerate(indices):
            values = np.asarray(data[band_position].filled(np.nan), dtype=np.float64)
            valid_values = values[footprint & np.isfinite(values)]
            n_valid_pixels = int(valid_values.size)
            pct_valid_pixels = (
                100.0 * n_valid_pixels / site_area_pixels if site_area_pixels else np.nan
            )
            valid_area_ha = n_valid_pixels * pixel_area_m2 / 10_000.0
            valid_area_km2 = n_valid_pixels * pixel_area_m2 / 1_000_000.0

            if n_valid_pixels:
                p25, p50, p75 = np.percentile(valid_values, [25.0, 50.0, 75.0])
                mean_value = float(np.mean(valid_values))
                std_value = float(np.std(valid_values, ddof=1)) if n_valid_pixels > 1 else np.nan
                p25, p50, p75 = float(p25), float(p50), float(p75)
            else:
                mean_value = std_value = p25 = p50 = p75 = np.nan

            records.append({
                "site": site,
                "year": year,
                "month": month,
                "window": window,
                "index": index_name,
                "n_valid_pixels": n_valid_pixels,
                "site_area_pixels": site_area_pixels,
                "pct_valid_pixels": pct_valid_pixels,
                "mean": mean_value,
                "std": std_value,
                "p25": p25,
                "p50": p50,
                "p75": p75,
                "total_area_ha": total_area_ha,
                "total_area_km2": total_area_km2,
                "valid_area_ha": valid_area_ha,
                "valid_area_km2": valid_area_km2,
                "n_scenes": int(qa_info["n_scenes"]),
                "mean_cloud": float(qa_info["mean_cloud"]),
            })

    return pd.DataFrame(records)

## 4 · Compute and Export Statistics to CSV

In [5]:
# Compute statistics across all composite rasters
stats_df = extract_imagery_statistics(
    tif_paths=tif_files,
    site_masks=site_masks,
    site_pixel_area=site_pixel_area,
    qa_manifest_path=QA_MANIFEST_PATH,
    indices=TARGET_INDICES,
)

# Export the complete statistics table
OUTPUT_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
stats_df.to_csv(OUTPUT_CSV_PATH, index=False)
print(f"Exported {len(stats_df)} records to {OUTPUT_CSV_PATH}")

Exported 784 records to /home/tshihadah/code/wadi-gaza-research/outputs/composites/imagery_stats.csv


## 5 · Data Preview and Quality Checks

In [6]:
# Preview structure and first records
print(f"Summary: {len(stats_df)} rows across {stats_df['site'].nunique()} sites, "
      f"{stats_df['window'].nunique()} composite windows, and {stats_df['index'].nunique()} indices.")

stats_df.head(12)

Summary: 784 rows across 7 sites, 28 composite windows, and 4 indices.


,site,year,month,window,index,n_valid_pixels,site_area_pixels,pct_valid_pixels,mean,std,p25,p50,p75,total_area_ha,total_area_km2,valid_area_ha,valid_area_km2,n_scenes,mean_cloud
0,AlAhrashProtectorate,2019,01,2019-01,NDVI,33848,33848,100.0,0.196705,0.130599,0.090658,0.144806,0.274628,338.0877,3.380877,338.0877,3.380877,5,5.740000
1,AlAhrashProtectorate,2019,01,2019-01,GNDVI,33848,33848,100.0,0.310632,0.113981,0.217765,0.265460,0.381605,338.0877,3.380877,338.0877,3.380877,5,5.740000
2,AlAhrashProtectorate,2019,01,2019-01,SAVI,33848,33848,100.0,0.166062,0.088229,0.090444,0.137430,0.227652,338.0877,3.380877,338.0877,3.380877,5,5.740000
3,AlAhrashProtectorate,2019,01,2019-01,NDRE,33848,33848,100.0,0.106442,0.089840,0.032901,0.074082,0.163031,338.0877,3.380877,338.0877,3.380877,5,5.740000
4,AlAhrashProtectorate,2019,05,2019-05,NDVI,33848,33848,100.0,0.134354,0.063699,0.081725,0.119690,0.170637,338.0877,3.380877,338.0877,3.380877,6,3.376667
5,AlAhrashProtectorate,2019,05,2019-05,GNDVI,33848,33848,100.0,0.268016,0.067253,0.211677,0.250833,0.311106,338.0877,3.380877,338.0877,3.380877,6,3.376667
6,AlAhrashProtectorate,2019,05,2019-05,SAVI,33848,33848,100.0,0.116600,0.043834,0.080787,0.109986,0.138469,338.0877,3.380877,338.0877,3.380877,6,3.376667
7,AlAhrashProtectorate,2019,05,2019-05,NDRE,33848,33848,100.0,0.078530,0.049521,0.035680,0.067596,0.112006,338.0877,3.380877,338.0877,3.380877,6,3.376667
8,AlAhrashProtectorate,2019,09,2019-09,NDVI,33848,33848,100.0,0.124300,0.060065,0.075915,0.110278,0.155746,338.0877,3.380877,338.0877,3.380877,5,3.064000
9,AlAhrashProtectorate,2019,09,2019-09,GNDVI,33848,33848,100.0,0.267136,0.062381,0.216385,0.251136,0.305129,338.0877,3.380877,338.0877,3.380877,5,3.064000


### Site-level completeness and cloud masking summary

In [7]:
# Check valid area coverage summary by site (using NDVI as representative index)
site_summary = stats_df[stats_df["index"] == "NDVI"].groupby("site").agg(
    n_composites=("window", "count"),
    total_area_ha=("total_area_ha", "first"),
    min_pct_valid=("pct_valid_pixels", "min"),
    mean_pct_valid=("pct_valid_pixels", "mean"),
    max_pct_valid=("pct_valid_pixels", "max"),
    mean_scenes=("n_scenes", "mean"),
).round(2)

site_summary

,n_composites,total_area_ha,min_pct_valid,mean_pct_valid,max_pct_valid,mean_scenes
site,,,,,,
AlAhrashProtectorate,28,338.09,100.00,100.00,100.0,4.46
BesorStream,28,113.03,100.00,100.00,100.0,4.46
HaBesorNationalPark,28,918.64,100.00,100.00,100.0,4.46
HolotNizzanim,28,1395.25,99.67,99.95,100.0,4.18
HolotPalmahim,28,620.89,100.00,100.00,100.0,4.18
NahalRubin,28,78.56,97.80,99.70,100.0,4.18
WadiGaza,28,284.33,99.86,99.99,100.0,4.50
